In [1]:
import glob, os, sys, time

# code is staged OFFLINE as a Kaggle Dataset — no GitHub clone (Internet can be OFF)
CODE = os.path.dirname(sorted(glob.glob("/kaggle/input/**/rank.py", recursive=True))[0])
sys.path.insert(0, CODE)

CAND = sorted(glob.glob("/kaggle/input/**/candidates.jsonl*", recursive=True))[0]
ART  = os.path.dirname(sorted(glob.glob("/kaggle/input/**/cand_embeddings.npy", recursive=True))[0])

print("code      :", CODE)
print("candidates:", CAND)
print("artifacts :", ART)
print("rank.py   :", os.path.isfile(CODE + "/rank.py"))

code      : /kaggle/input/datasets/nishanroy/redrob-ranker-code
candidates: /kaggle/input/datasets/nishanroy/redrob-candidates/candidates.jsonl
artifacts : /kaggle/input/notebooks/nishanroy/redrob-ai-challenge/artifacts
rank.py   : True


In [2]:
os.chdir(CODE)
t0 = time.time()
!python rank.py --candidates "{CAND}" --artifacts "{ART}" --out /kaggle/working/submission.csv
print(f"\nrank.py wall time: {time.time()-t0:.1f}s  (budget: 300s)")

Loading candidate pool...
  100000 candidates (7.7s)
Loading semantic similarity...
Scoring...
Writing 100 rows -> /kaggle/working/submission.csv
Done in 111.2s. Honeypot-flagged in top 100: 0

rank.py wall time: 112.7s  (budget: 300s)


In [3]:
!python "{CODE}/validate_submission.py" /kaggle/working/submission.csv

Submission is valid.


In [4]:
import csv
rows = list(csv.DictReader(open("/kaggle/working/submission.csv", encoding="utf-8")))
scores = [float(r["score"]) for r in rows]
ids = set(r["candidate_id"] for r in rows)

print("Top 10:")
for r in rows[:10]:
    print(f"  {r['rank']:>3} {r['candidate_id']} {r['score']}  {r['reasoning'][:95]}")

print(f"\nStuffer CAND_0000083 in top 100? {'CAND_0000083' in ids}")
print(f"Score range: {scores[0]:.4f} -> {scores[-1]:.4f}   unique scores: {len(set(scores))}")
print(f"Unique reasonings: {len(set(r['reasoning'] for r in rows))}/100")

Top 10:
    1 CAND_0077337 0.9817  Staff Machine Learning Engineer with 7.0 years — built recommendation systems at Paytm; hands-o
    2 CAND_0065878 0.9814  7.8-year Senior Data Scientist — built recommendation systems at Uber; hands-on Recommendation 
    3 CAND_0044222 0.9809  AI Engineer, 7.7y — built recommendation systems; hands-on Fine-tuning LLMs; strong, available 
    4 CAND_0046525 0.9805  6.1-year Senior Machine Learning Engineer — worked on ranking systems at LinkedIn; hands-on Inf
    5 CAND_0050454 0.9804  AI Engineer with 6.8 years — built recommendation systems; hands-on BM25; strong, available mat
    6 CAND_0062247 0.9796  AI Engineer with 7.3 years — built recommendation systems; hands-on Hugging Face Transformers; 
    7 CAND_0018499 0.9795  7.2-year Senior Machine Learning Engineer — worked on ranking systems at Zomato; hands-on Recom
    8 CAND_0005649 0.9792  Senior Data Scientist with 7.4 years — built recommendation systems; hands-on Information Retri
    9 CA

In [5]:
import json, gzip, collections
opener = gzip.open if CAND.endswith(".gz") else open
pool = {}
with opener(CAND, "rt", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            c = json.loads(line)
            pool[c["candidate_id"]] = c["profile"]["current_title"]

tc = collections.Counter(pool[r["candidate_id"]] for r in rows)
adj = sum(n for t, n in tc.items() if any(k in t.lower() for k in
          ["search", "software", "backend", "data engineer", "platform", "full stack"]))
print(f"Adjacent/plain-language titles now in top 100: {adj}\n")
for t, n in tc.most_common(15):
    print(f"  {n:3d}  {t}")

Adjacent/plain-language titles now in top 100: 5

   21  ML Engineer
   19  Data Scientist
   13  Junior ML Engineer
    7  AI Engineer
    6  Machine Learning Engineer
    5  Senior Data Scientist
    5  Search Engineer
    5  Applied ML Engineer
    5  NLP Engineer
    4  Senior Machine Learning Engineer
    3  Staff Machine Learning Engineer
    3  Senior NLP Engineer
    2  Senior AI Engineer
    1  Senior Applied Scientist
    1  Recommendation Systems Engineer
